In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Concatenate, Dropout
from tensorflow.keras.optimizers import Adam

class AiviewDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, video_names, labels, vis_dir, aud_dir, batch_size=32):
        self.video_names = video_names
        self.labels = labels # Ini adalah Y (Ground Truth)
        self.vis_dir = vis_dir
        self.aud_dir = aud_dir
        self.batch_size = batch_size

    def __len__(self):
        return int(np.ceil(len(self.video_names) / self.batch_size))

    def __getitem__(self, index):
        batch_videos = self.video_names[index * self.batch_size:(index + 1) * self.batch_size]
        batch_labels = self.labels[index * self.batch_size:(index + 1) * self.batch_size]
        
        vis_features = []
        aud_features = []
        
        for name in batch_videos:
            # Load X (Input)
            vis = np.load(f"{self.vis_dir}/{name}.npy") # (30, 4096)
            aud = np.load(f"{self.aud_dir}/{name}.npy") # (15, 128)
            vis_features.append(vis)
            aud_features.append(aud)
            
        return [np.array(vis_features), np.array(aud_features)], np.array(batch_labels)

I0000 00:00:1779033455.931702    3331 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779033456.594698    3331 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779033458.406204    3331 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
def build_aiview_model():
    # Jalur Visual (VGG-Face) - Menangani 4096 dimensi
    vis_input = Input(shape=(30, 4096), name='visual_input')
    vis_lstm = Bidirectional(LSTM(64, return_sequences=False))(vis_input)
    vis_drop = Dropout(0.3)(vis_lstm)

    # Jalur Audio (VGGish) - Menangani 128 dimensi
    aud_input = Input(shape=(15, 128), name='audio_input')
    aud_lstm = Bidirectional(LSTM(32, return_sequences=False))(aud_input)
    aud_drop = Dropout(0.3)(aud_lstm)

    # Late Fusion - Menggabungkan representasi laten
    merged = Concatenate()([vis_drop, aud_drop])
    
    # Fully Connected Layers untuk mempelajari korelasi
    dense1 = Dense(64, activation='relu')(merged)
    dense2 = Dense(32, activation='relu')(dense1)
    
    # Output Layer - 5 dimensi OCEAN dengan Sigmoid
    output = Dense(5, activation='sigmoid', name='ocean_output')(dense2)

    model = Model(inputs=[vis_input, aud_input], outputs=output)
    return model

model = build_aiview_model()
model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse', metrics=['mae'])
model.summary()

I0000 00:00:1779033461.737269    3331 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ visual_input        │ (None, 30, 4096)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ audio_input         │ (None, 15, 128)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │  2,130,432 │ visual_input[0][… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 64)        │     41,216 │ audio_input[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 192)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │     12,352 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ocean_output        │ (None, 5)         │        165 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,186,245 (8.34 MB)

 Trainable params: 2,186,245 (8.34 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Asumsi kamu sudah membagi video_names dan labels_ocean
# dari file .pkl dataset First Impressions
train_gen = AiviewDataGenerator(train_videos, train_labels, 
                                "D:/AIVIEW/features/visual", 
                                "D:/AIVIEW/features/audio")

val_gen = AiviewDataGenerator(val_videos, val_labels, 
                              "D:/AIVIEW/features/visual", 
                              "D:/AIVIEW/features/audio")

# Mulai melatih Bi-LSTM
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint('best_aiview_model.h5', save_best_only=True)
    ]
)